# Fetch Encounter Ranking (How-to-use)

1. **Set Parameters:**  
   Use the widgets to select `encounter_id`, `debug` mode, and `pages`.

2. **Run Notebook:**  
   Just hit "Run All" after setting parameters. The following happens:
   - Imports dependencies and settings.
   - Fetches and combines reports for the given encounter and number of pages. 
   - Saves results as JSON and uploaded to according volume, set in settings.
   - Output are printed and saved for analysis.

3. **Debug Mode:**  
   If `debug` is `True`, uses a sample payload.

## Import Dependencies, Job Settings, Schemas, and Helpers

1. Imports all required Python libraries
2. Add and import custom job settings, schema definitions, and helper functions.

In [0]:
import asyncio
import json
from datetime import datetime
from httpx import AsyncClient
from typing import Annotated, Callable, ClassVar

import pandas as pd
import numpy as np
from pydantic import (
    BaseModel, 
    field_validator, 
    Field,
    model_serializer
)
from unittest.mock import patch, AsyncMock, MagicMock

In [0]:
%reload_ext autoreload

from schemas import GraphQLQuery
from settings import fflogs_settings as settings

## Seting this notebook's parameters

In [0]:
def define_task_widget():
    dbutils.widgets.dropdown(
        name="debug",
        defaultValue="True",
        choices=["True", "False"],
        label="Debug mode to run notebook-wise."
    )
    dbutils.widgets.dropdown(
        name="zone_id", 
        defaultValue="76", 
        choices=["76"], 
        label="Zone ID to fetch reports from"
    )
    dbutils.widgets.text(
        name="guild_id",
        defaultValue="140098",
        label="Guild ID to fetch reports from"
    )
    dbutils.widgets.text(
        name="offset",
        defaultValue="0",
        label="Page to start pulling from"
    )
    dbutils.widgets.dropdown(
        name="limit",
        defaultValue="1",
        choices=["1", "5", "10"],
        label="Number of pages to fetch"
    )
    dbutils.widgets.dropdown(
        name="per_page_records",
        defaultValue="100",
        choices=["50", "100", "200"],
        label="Number of records per page"
    )

## Wipes by Encounter
1. Rate limit query to check API throttle. 
2. Reports query to fetch reports per zone.
3. Death events query to check the death events per encounter.
3. A sample of the expected successful response payload (also usable later in `debug` mode to test flow)

In [0]:
class RateLimitQuery(GraphQLQuery):
    QUERY: ClassVar[str] = """
    query RateLimitQuery {
        rateLimitData {
            limitPerHour
            pointsSpentThisHour
            pointsResetIn
        }
    }
    """

In [0]:
RATE_LIMIT_JSON = {
    'data': {
        'rateLimitData': {
            'limitPerHour': 3600, 
            'pointsSpentThisHour': 3, 
            'pointsResetIn': 3289
        }
    }
}

In [0]:
class EncounterReportsListQuery(GraphQLQuery):
    QUERY: ClassVar[str] = """
    query GetReportsByZone(
      $zoneID: Int
      $guildID: Int
      $startTime: Float
      $endTime: Float
      $limit: Int
      $page: Int
      $encounterID: Int
    ) {
      reportData {
        reports(
          zoneID: $zoneID
          guildID: $guildID
          startTime: $startTime
          endTime: $endTime
          limit: $limit
          page: $page
        ) {
          total
          last_page
          has_more_pages
          
          data {
            code
            title
            fights(encounterID: $encounterID) {
              id
              encounterID
              kill
              startTime
              endTime
              difficulty
              size
              fightPercentage
              bossPercentage
              lastPhase
              lastPhaseIsIntermission
              wipeCalledTime
              friendlyPlayers
            }
          }
        }
      }
    }
    """

    zone_id: int
    guild_id: int 
    start_time: str | float
    end_time: str | float
    limit: int
    page: int
    encounter_id: int | None

In [0]:
ENCOUNTER_RANKING_JSON = {
    "worldData": {
        "encounter": {
            "id": 1,
            "name": "Futures Rewritten (Ultimate)",
            "zone": {
                "id": 56,
                "name": "Ultimates (Dawntrail)"
            },
            "characterRankings": {
                "rankings": [
                    {
                        "name": "Lyra Starweaver",
                        "spec": "WhiteMage",
                        "amount": 12453.67,
                        "aDPS": 11980.23,
                        "startTime": 1717200000000,
                        "duration": 485000,
                        "lodestoneID": 12345678,
                        "guild": {
                            "id": 16,
                            "name": "Please Be Careful"
                        },
                        "server": {
                            "id": 8,
                            "name": "Gilgamesh"
                        }
                    }
                ]
            }
        }
    }
}

## Send Query to Endpoint & Batch Results

1. Send a GraphQL query to the endpoint asynchronously
2. Batch multiple queries with `asyncio.gather` concurrency to combine multiple pages.
3. Write results to volume

In [0]:
async def query_graphql(
    client: AsyncClient,
    query: GraphQLQuery,
    access_token: str,
    record_key: str = "data",
) -> dict:
    """Executes a GraphQL query against the configured endpoint asynchronously.

    Args:
        client (AsyncClient): The httpx async client to use for the request.
        query (GraphQLQuery): The GraphQL query object to execute.
        access_token (str): Bearer token used to authenticate the request.
        record_key (str, optional): If provided, returns response.json()[record_key]
            instead of the full response JSON. Defaults to None.

    Returns:
        dict: The full response JSON, or the value at response.json()[record_key]
            if record_key is specified.

    Raises:
        ValueError: If the GraphQL response contains errors.
        httpx.HTTPStatusError: If the HTTP request fails.
    """
    _FFLOGS_QUERY_ENDPOINT = "https://www.fflogs.com/api/v2/client"

    response = await client.post(
        _FFLOGS_QUERY_ENDPOINT,
        json=query.model_dump(),
        headers={
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json",
        },
    )
    response.raise_for_status()
    payload = response.json()

    if "errors" in payload:
        raise ValueError(f"GraphQL errors: {payload['errors']}")

    return payload[record_key] if record_key is not None else payload

In [0]:
async def batch_query_graphql(
    client: AsyncClient,
    encounter_id: int,
    access_token: str,
    pages: int = 10,
) -> dict:
    """Fetches and aggregates encounter rankings across multiple pages asynchronously.

    Args:
        client (AsyncClient): The httpx async client to use for requests.
        encounter_id (int): The ID of the encounter to query.
        access_token (str): Bearer token used to authenticate the request.
        pages (int, optional): Number of pages to fetch. Defaults to 10.

    Returns:
        dict: The aggregated GraphQL response with all rankings combined.

    Raises:
        ValueError: If any GraphQL response contains errors.
        httpx.HTTPStatusError: If any HTTP request fails.
    """
    queries = [
        EncounterRankingQuery(id=encounter_id, page=page) 
        for page in range(1, pages + 1)
    ]
    results = await asyncio.gather(*[
        query_graphql(client, query, access_token)
        for query in queries
    ])

    all_rankings = []
    for data in results:
        all_rankings.extend(
            data["worldData"]["encounter"]["characterRankings"]["rankings"]
        )

    results[0]["worldData"]["encounter"]["characterRankings"]["rankings"] = all_rankings
    
    return results[0]

In [0]:
def write_json_to_volume(
    data: dict, 
    volume_dir: str,
    file_name: str
) -> None:
    now = datetime.now().strftime("%Y%m%d_%H%M%S")

    with open(f"{volume_dir}/{file_name}_{now}.json", "w") as f:
        json.dump(data, f, indent=4)
        

## Orchestrate All Tasks & Debug/Test Mode

1. Coordinates the entire workflow
2. Running test functions when debug mode is enabled.

In [0]:
async def rate_limit_check() -> None:
    rate_limit_query = RateLimitQuery()
    async with AsyncClient() as client:
        rate_limit_response = await query_graphql(
            client=client,
            query=rate_limit_query,
            access_token=settings.fflogs_access_token,
            record_key=None
        )

    print(json.dumps(rate_limit_response, indent=4))

In [0]:
async def query_reports_to_volume() -> None:
    # ----- Capture widget variables
    define_task_widget()
    is_debug = dbutils.widgets.get("debug") == "True"
    zone_id = int(dbutils.widgets.get("zone_id"))
    guild_id = int(dbutils.widgets.get("guild_id"))

    offset = int(dbutils.widgets.get("offset"))
    per_page_records = int(dbutils.widgets.get("per_page_records"))
    page_limits = int(dbutils.widgets.get("page_limit"))

    # ----- Query reports per zone
    zone_reports_query = EncounterReportsListQuery(
        zone_id=zone_id,
        guild_id=guild_id,
        start_time="2026-06-06 00:00:00",
        limit=per_page_records,
        page=1,
    )
    async with AsyncClient() as client:
        zone_reports_response = await query_graphql(
            client=client,
            query=zone_reports_query,
            access_token=settings.fflogs_access_token,
            record_key="data"
        )
        limit_response = await rate_limit_check()

        print(json.dumps(zone_reports_response, indent=4))
        print(json.dumps(limit_response, indent=4))


In [0]:
await query_reports_to_volume()

In [0]:
async def test_query_encounter_to_volume() -> None:
    mock_client = AsyncMock(spec=AsyncClient)
    with (
        patch("httpx.AsyncClient.__aenter__", return_value=mock_client),
        patch("httpx.AsyncClient.__aexit__", return_value=False),
        patch("__main__.batch_query_graphql",
        new=AsyncMock(return_value=ENCOUNTER_RANKING_JSON)) as mock_batch,
        patch("__main__.write_json_to_volume") as mock_write,
    ):
        await query_encounter_to_volume()

        mock_batch.assert_awaited_once_with(
            client=mock_client,
            encounter_id=int(dbutils.widgets.get("encounter_id")),
            access_token=settings.fflogs_access_token,
            pages=int(dbutils.widgets.get("pages"))
        )
        mock_write.assert_called_once_with(
            data=ENCOUNTER_RANKING_JSON,
            volume_dir=settings.volume_dir,
            file_name=f"encounter_{dbutils.widgets.get('encounter_id')}"
        )
        print("✓ test_query_encounter_to_volume passed")

In [0]:
if dbutils.widgets.get("debug") == "True":
    await test_query_encounter_to_volume()
else:
    await query_encounter_to_volume()